# Stage 6 — Per-Prediction Confidence Tiers

Attaches an honest, rule-based confidence tier to every (gene, cell-line)
prediction. Confidence is **categorical and deterministic** — not a fabricated
0–1 score. The three inputs are:

| Input | Source |
|---|---|
| `n_layers` | `core_score.parquet` — 1 or 2 omics layers scored |
| `regime` + `regime_source` | `gene_regime.parquet` — measured / unknown |
| `rank_basis` | derived in Stage 4 — core_score / driver_flag+score / score_only |

**Tier rules:**
- **high** — abundance_tracking + measured + 2 layers
- **moderate** — measured regime + clear rank basis (core_score or driver_flag+score)
- **low** — activation_driven + score_only (no driver alteration in this line)
- **unknown** — regime not measured (no validation data)

Writes **`outputs/predictions_with_confidence.parquet`**.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

OUTPUTS = Path("outputs")

In [ ]:
core   = pd.read_parquet(OUTPUTS / "core_score.parquet")
regime = pd.read_parquet(OUTPUTS / "gene_regime.parquet")
flags  = pd.read_parquet(OUTPUTS / "flags_with_driver.parquet")
gdsc   = pd.read_parquet("../../validation/prepared/gdsc_scored_ready.parquet",
                         columns=["model_id","target_ensg","sensitive"])

core["model_id"]  = core["model_id"].str.lower()
flags["model_id"] = flags["model_id"].str.lower()
gdsc["model_id"]  = gdsc["model_id"].str.lower()
gdsc["target_ensg"] = gdsc["target_ensg"].astype("string").str.lower()

print("core cols:",   core.columns.tolist())
print("regime cols:", regime.columns.tolist())
print("n_layers:",    core.n_layers.value_counts().to_dict())
print("regime_source:", regime.regime_source.value_counts().to_dict())

In [ ]:
# Curated genes: intersection of regime and GDSC targets
curated_genes = sorted(set(regime.ensg_id) & set(gdsc.target_ensg))
print(f"Curated genes: {len(curated_genes)}")

In [ ]:
# Filter to curated genes and join all signals in one pass
core_c  = core[core.ensg_id.isin(curated_genes)].copy()
flags_c = flags[flags.ensg_id.isin(curated_genes)][["model_id","ensg_id","has_driver_alteration"]].copy()

core_c = core_c.merge(regime[["ensg_id","class","regime_source"]], on="ensg_id", how="left")
core_c["regime_source"] = core_c["regime_source"].fillna("unknown")
core_c["class"]         = core_c["class"].fillna("unknown")

core_c = core_c.merge(flags_c, on=["model_id","ensg_id"], how="left")
core_c["has_driver_alteration"] = core_c["has_driver_alteration"].fillna(False)

# rank_basis (vectorised)
core_c["rank_basis"] = np.where(
    core_c["class"] == "abundance_tracking", "core_score",
    np.where(core_c["has_driver_alteration"], "driver_flag+score", "score_only")
)

print("rank_basis distribution:")
print(core_c["rank_basis"].value_counts().to_string())

In [ ]:
# Confidence tier (vectorised, rule-based)
tier = pd.Series("moderate", index=core_c.index)
tier = np.where(
    (core_c.regime_source == "measured") & (core_c["class"] == "abundance_tracking") & (core_c.n_layers == 2),
    "high", tier)
tier = np.where(
    (core_c["class"] == "activation_driven") & (core_c.rank_basis == "score_only"),
    "low", tier)
tier = np.where(core_c.regime_source != "measured", "unknown", tier)
core_c["confidence"] = tier

print("Confidence tier distribution:")
print(core_c["confidence"].value_counts().to_string())
print()
print("By regime class:")
print(core_c.groupby(["class","confidence"]).size().unstack(fill_value=0).to_string())

In [ ]:
def confidence_reason(row):
    if row["confidence"] == "high":
        return (f"High: {row['class']} gene (empirically validated), "
                f"scored on {int(row['n_layers'])} layers.")
    if row["confidence"] == "low":
        return ("Low: activation-driven gene dependency; ranked on score alone "
                "with no supporting driver alteration in this line.")
    if row["confidence"] == "unknown":
        return ("Unknown: no validation data for this gene; scoring regime "
                "unclassified, ranking shown but not regime-weighted.")
    if row["rank_basis"] == "driver_flag+score":
        return ("Moderate: activation-driven gene; ranked on driver alteration "
                "present in this line, with score as support.")
    return (f"Moderate: {row['class']} gene, ranked on {row['rank_basis']}, "
            f"{int(row['n_layers'])}-layer coverage.")

core_c["confidence_reason"] = core_c.apply(confidence_reason, axis=1)

# Sanity sample
for tier in ["high", "moderate", "low"]:
    print(f"--- {tier} ---")
    print(core_c[core_c.confidence == tier][["model_id","ensg_id","core_score","n_layers",
                                              "rank_basis","confidence","confidence_reason"]]
          .head(2).to_string(index=False))
    print()

In [ ]:
# Biology sanity check: BCL2 (abundance) vs BRAF (activation_driven)
for name, ensg in [("BCL2","ensg00000171791"), ("BRAF","ensg00000157764")]:
    sub = core_c[core_c.ensg_id == ensg]
    if sub.empty:
        print(f"{name}: not in curated genes")
    else:
        print(f"{name} ({sub['class'].iloc[0]}): {sub['confidence'].value_counts().to_dict()}")
# Expected: BCL2 high/moderate; BRAF mostly low with moderate where driver flag fires

In [ ]:
out_cols = ["model_id","ensg_id","core_score","n_layers","stratum_rank",
            "class","regime_source","rank_basis","has_driver_alteration",
            "confidence","confidence_reason"]
out = core_c[out_cols]
out.to_parquet(OUTPUTS / "predictions_with_confidence.parquet", index=False)
print("Written: outputs/predictions_with_confidence.parquet")
print("Shape:", out.shape)
print("Columns:", out.columns.tolist())